<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c08-header.png" alt="Nextia Learning · Building Reliable Applications with LLM APIs" width="100%">

# Classify banking requests with an LLM

**[Classify banking requests with an LLM](https://learning.nextia-ai.com/courses/llm-apps/m07/classify-banking-requests/)** · Building Reliable Applications with LLM APIs · Module 7, case study 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** route real bank-support messages to 77 intents with a hosted language model, a prompt and a strict schema, and **no training examples**. You compare two prompt versions on a fixed validation set, test the chosen one once, and compare it honestly with the deep learning course's trained routers: accuracy with an interval, the errors, the cost and latency per 1,000 messages, and what data leaves the bank.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page. The notebook runs in under a minute. |
| **Needs** | An internet connection for the setup cells (about 4 MB). No account, no key, no graphics card: every model answer is a real recording. |
| **Recorded** | gpt-6-luna as `chat-small` and gpt-6.1-sol as `chat-strong`, on Azure. `chat-small` answered all 3,080 test messages; `chat-strong` answered a fixed sample of 770 (10 per intent), to stay within the case study's budget. |
| **You should know** | [Compare prompts](https://learning.nextia-ai.com/courses/llm-apps/m02/compare-prompts/), [Define a schema](https://learning.nextia-ai.com/courses/llm-apps/m03/define-a-schema/) and [Validate meaning](https://learning.nextia-ai.com/courses/llm-apps/m03/validate-meaning/) in this course. The deep learning course's case study [Fine-tune a small transformer](https://learning.nextia-ai.com/courses/deep-learning/m09/fine-tune-a-small-transformer/) helps, but you do not need it. |
| **Data** | BANKING77 (PolyAI, 2020), CC BY 4.0: 13,083 short messages to a bank's online support, 77 intents. |
| **Tested** | Python 3.12 with pydantic 2.14.0, scikit-learn 1.9.1 and pandas, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/llm-apps/m07/classify-banking-requests/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C08/M07-L03-classify-banking-requests/classify-banking-requests-solution.ipynb).

## Setup: packages

Run the next cell. It uses `pydantic`, `scikit-learn` and `pandas`, which Colab and Kaggle already have, and installs any that is missing. You should see a line with the versions. The notebook needs no account and no key: every model answer comes from a recording.

In [ ]:
import importlib, platform, subprocess, sys
for module, package in (("pydantic", "pydantic>=2"), ("sklearn", "scikit-learn"), ("pandas", "pandas")):
    try:
        importlib.import_module(module)
    except ImportError:  # not installed: install it
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
import pydantic, sklearn, pandas
print("Python", platform.python_version(), "| pydantic", pydantic.__version__, "| scikit-learn", sklearn.__version__,
      "| pandas", pandas.__version__)

## Setup: the data and the recorded answers

Run the next cell. It makes a folder `banking-intents` and downloads into it:

- the two files of **BANKING77** (about 1.1 MB) from the publisher's GitHub repository, at one fixed version;
- the recorded answers of the two models and the two prompt versions (about 3 MB) from the course's labs repository.

It checks the checksum of every file. You should see `Ready: train.csv 10,003 messages, test.csv 3,080 messages, 6 recordings.`

If the cell shows an error, read its last line. `Could not download`: check your internet connection and run the cell again (in Kaggle, turn on **Internet** in the settings). `wrong checksum`: delete the folder `banking-intents` and run the cell again.

Data: "BANKING77" by PolyAI (Casanueva et al., 2020), https://github.com/PolyAI-LDN/task-specific-datasets, licensed under CC BY 4.0. We changed nothing in the files.

In [ ]:
import csv, hashlib, json, os, subprocess, urllib.request
from pathlib import Path

BANKING = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/57ec275d8078af65b7731c2a98be812d844a6d6b/banking_data/"
LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C08/M07-L03-classify-banking-requests/")
FILES = {
    "data/train.csv": (BANKING + "train.csv", "b06e26ac675513959a63135f11b94ea7786ed02da65db93a5650d8838cbc664b"),
    "data/test.csv": (BANKING + "test.csv", "d12d6e3bc4c3103966ae786dc435913c0c563dfa328f5a3646d0e62cfeeb474d"),
    "recordings/valid1000_chat-small_v1.jsonl": (LABS + "recordings/valid1000_chat-small_v1.jsonl", "52d9a7fdddf2fd945186bc1573fe7fea2b7dc87f4235a46a5484ab9941f2f1a7"),
    "recordings/valid1000_chat-small_v2.jsonl": (LABS + "recordings/valid1000_chat-small_v2.jsonl", "3e4a8f0e5010747ac545fb4e437cdaa429fc80b51eb0213741be0547bb5924a3"),
    "recordings/valid300_chat-small_v1_text.jsonl": (LABS + "recordings/valid300_chat-small_v1_text.jsonl", "4df120ddba84cc18227a2f6324089b21d16fa931d0404b830888b13cfe26c22f"),
    "recordings/test_chat-small_v2.jsonl": (LABS + "recordings/test_chat-small_v2.jsonl", "79debe27b6ea18d0ade57f55b57bfe71ca9f1f21509da79617a3cd53ef6b812c"),
    "recordings/test770_chat-strong_v2.jsonl": (LABS + "recordings/test770_chat-strong_v2.jsonl", "5cf44d65a4cab4ed8151646d5e5620bf2e03435966b8959d51c831ad6b79ee07"),
    "recordings/new_messages.jsonl": (LABS + "recordings/new_messages.jsonl", "d745a2af5958734a4340aa88a6a06621c713e9fd17030b03c5a018c496752069"),
    "prompts/v1.md": (LABS + "prompts/v1.md", "b0771bcff0b169001e7c80c1ff18f75e768fc83713beb29d0d1ec6625bb76ade"),
    "prompts/v2.md": (LABS + "prompts/v2.md", "75b24586550de67cc51206f3a26a3440e5edc3c33cbad63f5591c3599c090dfd"),
    "prompts/intents.tsv": (LABS + "prompts/intents.tsv", "bb55445822574f167524e03d8c23a1754dc1abbf71a00c0b18433ff6c1012de3"),
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
WORK = Path("banking-intents").resolve()
for folder in ("data", "recordings", "prompts"):
    (WORK / folder).mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, (url, expected) in FILES.items():
    path = WORK / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(url, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder banking-intents and run again.")
os.chdir(WORK)

def read_csv(path):
    with open(path, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

def load_answers(name):
    """One recording: a list of dicts, one per message (see dataset.md)."""
    return [json.loads(line) for line in Path(f"recordings/{name}.jsonl").read_text(encoding="utf-8").splitlines()]

raw_train, raw_test = read_csv("data/train.csv"), read_csv("data/test.csv")
print(f"Ready: train.csv {len(raw_train):,} messages, test.csv {len(raw_test):,} messages, "
      f"{len(list(Path('recordings').glob('*.jsonl')))} recordings.")

## Setup: helpers

The next cell defines the check helpers and the 95% Wilson interval of the language models course's [How sure can 24 tickets be?](https://learning.nextia-ai.com/courses/llms/m04/measure-practical-performance/#how-sure-can-24-tickets-be):

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `wilson(k, n)` gives the 95% interval for k right out of n.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def wilson(k, n, z=1.96):
    """95% Wilson interval for k right out of n, as (low, high)."""
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * ((p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5) / (1 + z * z / n)
    return centre - half, centre + half

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data and the split

The split is the deep learning course's: 1,000 messages of the official train file are held out as the **validation set**, with the same share of each intent. The prompts are compared on them. The official test file stays sealed until section 5.

Run the next cell. You should see `9003 train | 1000 valid | 3080 test | 77 intents`, then two training messages for each of three intents that are easy to confuse.

In [ ]:
from sklearn.model_selection import train_test_split

train, valid = train_test_split(raw_train, test_size=1000, stratify=[r["category"] for r in raw_train], random_state=0)
test = raw_test
print(len(train), "train |", len(valid), "valid |", len(test), "test |", len({r["category"] for r in raw_train}), "intents")
for intent in ("card_arrival", "card_delivery_estimate", "get_physical_card"):
    print(f"{intent:24}", [r["text"] for r in train if r["category"] == intent][:2])

> **Check.** `get_physical_card` is mostly about the PIN of a new card, not about getting a card. A name is not a definition. Prompt v2 adds one line per intent for this reason.

## 2. The prompt and the schema

The next cell builds the request as the course project does: the prompt file is the system message, the customer's text goes between `<message>` tags, and the schema goes in `response_format` with `strict: true`. The `intent` field is an **enum** of the 77 intents, made from the same list as the prompt.

Run it. You should see `77 intents | 77 values in the enum` and the user message for one validation message.

In [ ]:
from typing import Literal
from pydantic import ConfigDict, ValidationError, create_model

INTENTS = [line.split("\t")[0] for line in Path("prompts/intents.tsv").read_text().splitlines()]

def message_block(text):
    """The customer's text between tags; tags inside the text are neutralised."""
    clean = text.replace("<message>", "(message)").replace("</message>", "(/message)")
    return f"<message>\n{clean}\n</message>"

def build_messages(text, version):
    return [{"role": "system", "content": Path(f"prompts/{version}.md").read_text()},
            {"role": "user", "content": message_block(text)}]

IntentAnswer = create_model(
    "IntentAnswer",
    __config__=ConfigDict(extra="forbid"),          # no other field allowed
    intent=(Literal[tuple(INTENTS)], ...),         # an enum of the 77 intents, required
    confidence=(Literal["low", "medium", "high"], ...),
)

def response_format():
    return {"type": "json_schema",
            "json_schema": {"name": "banking_intent", "strict": True, "schema": IntentAnswer.model_json_schema()}}

def build_request(text, model, version, schema=True):
    request = {"model": model, "messages": build_messages(text, version), "max_completion_tokens": 1000}
    if schema:
        request["response_format"] = response_format()
    return request

request = build_request(valid[0]["text"], "chat-small", "v2")
enum = request["response_format"]["json_schema"]["schema"]["properties"]["intent"]["enum"]
print(len(INTENTS), "intents |", len(enum), "values in the enum | first:", enum[:3])
print("system message:", len(request["messages"][0]["content"]), "characters | user message:")
print(request["messages"][1]["content"])

Look at the two prompt files: `prompts/v1.md` lists the 77 names only; `prompts/v2.md` adds one line per intent. The lines were written by the course from the intent names and training messages, not from validation or test messages.

Run the next cell to see the two versions of the same intents.

In [ ]:
v1_text, v2_text = Path("prompts/v1.md").read_text(), Path("prompts/v2.md").read_text()
for name in ("card_arrival", "card_delivery_estimate"):
    print("v1:", next(line for line in v1_text.splitlines() if line.startswith(f"- {name}")))
    print("v2:", next(line for line in v2_text.splitlines() if line.startswith(f"- {name}")))

## 3. Gate 1 on every recorded answer

Structured output keeps the answer in the schema's shape, but the course's rule is to [validate anyway](https://learning.nextia-ai.com/courses/llm-apps/m03/define-a-schema/#validate-anyway). The next cell checks every recorded answer with the same Pydantic model.

Run it. With the schema, every answer should be `valid`. The run without the schema (`_text`) comes back in section 9.

In [ ]:
from collections import Counter

def parse(text, finish_reason):
    """Gate 1: (answer, "") or (None, why). Checked in code even with strict structured output."""
    if finish_reason == "length":
        return None, "truncated"
    if not text:
        return None, "empty"
    try:
        return IntentAnswer.model_validate_json(text).model_dump(), ""
    except ValidationError as error:
        return None, "malformed_json" if error.errors()[0]["type"] == "json_invalid" else "schema"

answers = {name: load_answers(name) for name in ['valid1000_chat-small_v1', 'valid1000_chat-small_v2', 'valid300_chat-small_v1_text', 'test_chat-small_v2', 'test770_chat-strong_v2']}
for name, rows in answers.items():
    checked = Counter(parse(r["text"], r["finish_reason"])[1] or "valid" for r in rows)
    print(f"{name:30} {len(rows):5} answers  {dict(checked)}")

## 4. Baselines, then two prompts on the same 1,000 messages

First the simple baselines on the validation set: always the most common intent, and the deep learning course's TF-IDF with logistic regression, trained on the 9,003 training messages (about 2 seconds). You should see `majority 1.9%` and `TF-IDF 88.0%`.

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

def tokenize(text):
    """The deep learning course's word tokens."""
    return re.findall(r"[a-z]+|[0-9]+|[^\sa-z0-9]", text.lower())

majority = Counter(r["category"] for r in train).most_common(1)[0][0]
tfidf = make_pipeline(TfidfVectorizer(token_pattern=None, tokenizer=tokenize, lowercase=False, min_df=1),
                      LogisticRegression(max_iter=3000, C=100)).fit([r["text"] for r in train], [r["category"] for r in train])
tfidf_valid = tfidf.predict([r["text"] for r in valid])
tfidf_test = tfidf.predict([r["text"] for r in test])
print(f"majority {sum(r['category'] == majority for r in valid) / len(valid):.1%} | "
      f"TF-IDF {sum(p == r['category'] for p, r in zip(tfidf_valid, valid)) / len(valid):.1%}")

Now the two prompts. `chat-small` answered all 1,000 validation messages with v1 and with v2. Each line of a recording has the true intent (`label`) and the model's answer (`intent`).

> **Your turn.** Write `accuracy(rows)`: the share of rows where `intent` equals `label`. Then run the cell. For v1 you should see 80.3%.

In [ ]:
def accuracy(rows):
    # Your code here: the share of rows whose "intent" equals their "label"
    return ...

v1, v2 = answers["valid1000_chat-small_v1"], answers["valid1000_chat-small_v2"]
print("v1", accuracy(v1), "| v2", accuracy(v2))

In [ ]:
expect('the accuracy of v2 on validation', accuracy(v2), 0.842, tolerance=1e-3)

Two totals do not show which messages changed. The next cell counts, message by message, where only one version is right, and the paired 95% interval of the difference (the method of the mathematics course's [Compare alternatives](https://learning.nextia-ai.com/courses/math/m05/compare-alternatives/#an-interval-for-the-difference)).

> **Predict.** v2 sends about 887 more input tokens per message. Will its gain be larger than the interval's half-width?

Run it. You should see `only v1 right 14 | only v2 right 53`.

In [ ]:
v1_by_index = {r["index"]: r for r in v1}
pairs_v = [(v1_by_index[b["index"]], b) for b in v2]   # the same message, answered by v1 and by v2
only_v1 = sum(1 for a, b in pairs_v if a["intent"] == a["label"] and b["intent"] != b["label"])
only_v2 = sum(1 for a, b in pairs_v if b["intent"] == b["label"] and a["intent"] != a["label"])
n = len(v1)
diff = (only_v2 - only_v1) / n
se = (only_v1 + only_v2 - (only_v2 - only_v1) ** 2 / n) ** 0.5 / n
print(f"only v1 right {only_v1} | only v2 right {only_v2} | v2 - v1 = {diff:+.1%} "
      f"(95% interval {diff - 1.96 * se:+.1%} to {diff + 1.96 * se:+.1%})")
for name, rows in (("v1", v1), ("v2", v2)):
    ins = sorted(r["input_tokens"] for r in rows)
    print(f"{name}: median {ins[len(ins) // 2]} input tokens per message")

> **Check.** The interval is above zero, so v2's gain is not noise on these 1,000 messages. The course chose **v2** for the test. Read the lesson page for the messages that changed.

## 5. The test, once

The chosen prompt went to the sealed test file once. `chat-small` answered all 3,080 messages. `chat-strong` costs about 19 times more per message, so it answered a fixed sample of 770 (10 per intent). For a fair comparison on that sample, the cell also scores `chat-small` and TF-IDF on the same 770 messages.

Run it. You should see a table with five rows. `chat-small` on all test messages: 87.5%.

In [ ]:
import pandas as pd

small, strong = answers["test_chat-small_v2"], answers["test770_chat-strong_v2"]
sample = {r["index"] for r in strong}
rows = []
def add(router, messages, right, note=""):
    low, high = wilson(right, messages)
    rows.append({"router": router, "messages": messages, "right": right, "accuracy": right / messages,
                 "95% low": low, "95% high": high, "note": note})

add("TF-IDF + logistic regression", len(test), sum(p == r["category"] for p, r in zip(tfidf_test, test)), "trained, 9,003 labels")
add("Fine-tuned DistilBERT", len(test), 2826, "quoted from the deep learning course")
add("chat-small, prompt v2", len(small), sum(r["intent"] == r["label"] for r in small), "no training examples")
add("chat-strong, prompt v2 (sample)", len(strong), sum(r["intent"] == r["label"] for r in strong), "10 per intent")
add("chat-small, same sample", len(sample), sum(r["intent"] == r["label"] for r in small if r["index"] in sample), "10 per intent")
pd.DataFrame(rows).round(3)

## 6. Where it goes wrong

The next cell counts the errors of `chat-small` on the test set by pair of intents, in either direction. Run it. The top pair should be `beneficiary_not_allowed` and `failed_transfer`, with 18 errors.

In [ ]:
pairs = Counter(tuple(sorted((r["label"], r["intent"]))) for r in small if r["intent"] != r["label"])
pd.DataFrame([{"intent A": a, "intent B": b, "errors": n} for (a, b), n in pairs.most_common(10)])

Do the language model and TF-IDF fail on the same messages? Run the next cell. It prints the four counts of the agreement table and three messages where only one of the two is right.

In [ ]:
tf = {i: p for i, p in enumerate(tfidf_test)}
table = Counter((r["intent"] == r["label"], tf[r["index"]] == r["label"]) for r in small)
print("both right", table[(True, True)], "| LLM only", table[(True, False)],
      "| TF-IDF only", table[(False, True)], "| both wrong", table[(False, False)])
for r in [r for r in small if r["intent"] != r["label"] and tf[r["index"]] == r["label"]][:3]:
    print(f"- {test[r['index']]['text']!r}\n  true {r['label']} | LLM {r['intent']} | TF-IDF {tf[r['index']]}")

## 7. Cost and latency per 1,000 messages

Every recorded answer has its tokens and its latency. The course's formula: cost per 1,000 = the mean of (input tokens × input price + output tokens × output price) × 1,000. The prices are Azure's, in US$ per million tokens (check the current price page). The formula ignores the discount for cached input, so it is an upper bound.

> **Your turn.** Write `cost_per_1000(rows, price_in, price_out)`. For `chat-small` on the test set you should see about US$0.229.

In [ ]:
PRICES = {"chat-small": (0.10, 0.50), "chat-strong": (2.00, 10.00)}   # US$ per million tokens. Check the current price page.

def cost_per_1000(rows, price_in, price_out):
    # Your code here: the mean cost of one message in US$, times 1,000
    return ...

for name, rows in (("chat-small", small), ("chat-strong", strong)):
    lat = sorted(r["latency_s"] for r in rows)
    print(f"{name:12} US${cost_per_1000(rows, *PRICES[name])} per 1,000 | latency median {lat[len(lat) // 2]:.2f} s, "
          f"p95 {lat[int(0.95 * (len(lat) - 1))]:.2f} s")

In [ ]:
expect('the cost per 1,000 messages of chat-strong (US$)', cost_per_1000(strong, *PRICES['chat-strong']), 4.239, tolerance=2e-3)

For comparison, the deep learning course measured on a CPU: TF-IDF about 0.2 ms per message, fine-tuned DistilBERT about 7.4 ms, with no price per call (the bank pays for its own server).

## 8. A person for unsure messages

The model writes its own `confidence`. It is not a measured probability ([Validate meaning](https://learning.nextia-ai.com/courses/llm-apps/m03/validate-meaning/#what-about-the-models-confidence)), but it may still sort the answers. A second signal is free: does TF-IDF give the same intent?

> **Your turn.** Count the test messages that a person would handle under the rule "confidence is not high", and the accuracy of the messages that are routed automatically. You should see that about 19.6% go to a person.

In [ ]:
to_person = ...      # Your code here: how many rows of `small` have a confidence other than "high"
auto = [r for r in small if r["confidence"] == "high"]
auto_accuracy = ...  # Your code here: accuracy(auto)
print(to_person, "to a person |", auto_accuracy, "right among the rest")

In [ ]:
expect('messages sent to a person', to_person, 603)
expect('accuracy of the automatic routes', auto_accuracy, 0.9334, tolerance=1e-3)

Run the next cell to compare four rules on the same test messages: no fallback, confidence, disagreement with TF-IDF, and either.

In [ ]:
rules = {"no fallback": lambda r: False,
         "confidence not high": lambda r: r["confidence"] != "high",
         "LLM and TF-IDF disagree": lambda r: r["intent"] != tf[r["index"]],
         "either": lambda r: r["confidence"] != "high" or r["intent"] != tf[r["index"]]}
out = []
for name, rule in rules.items():
    rest = [r for r in small if not rule(r)]
    out.append({"rule": name, "to a person": len(small) - len(rest), "share": (len(small) - len(rest)) / len(small),
                "routed automatically": len(rest), "accuracy of those": accuracy(rest),
                "wrong and automatic": len(rest) - sum(r["intent"] == r["label"] for r in rest)})
pd.DataFrame(out).round(3)

## 9. Does the schema fix a problem you can see?

Before Module 3, the prompt only **asked** for a valid name. The recording `valid300_chat-small_v1_text` has the same v1 prompt **without** `response_format`, on 300 validation messages. Run the next cell. It counts the answers that are not JSON or not one of the 77 intents, and compares the accuracy with the schema run on the same 300 messages.

In [ ]:
text_rows = answers["valid300_chat-small_v1_text"]
with_schema = {r["index"]: r for r in answers["valid1000_chat-small_v1"]}
invented = Counter()
for r in text_rows:
    try:
        name = json.loads(r["text"]).get("intent")
    except (json.JSONDecodeError, TypeError, AttributeError):
        name = "(not JSON)"
    if name not in INTENTS:
        invented[name] += 1
print(sum(invented.values()), "of", len(text_rows), "answers are not one of the 77 intents", dict(invented))
print("right without the schema:", sum(r["intent"] == r["label"] for r in text_rows),
      "| with the schema, same messages:", sum(with_schema[r["index"]]["intent"] == r["label"] for r in text_rows))
ins = sorted(r["input_tokens"] for r in text_rows)
print("median input tokens without the schema:", ins[len(ins) // 2])

> **Check.** On this task, the model did not invent a name, and the accuracy is about the same. The schema costs about 460 input tokens per message and buys a **guarantee**: the code that routes the message can never get a name without an answer flow. Read the failure case on the lesson page for the limit on its size.

## 10. New messages, end to end

These messages are **made up** for this case study; no customer wrote them. The first six are the deep learning course's new messages, so you can compare with its fine-tuned model. Each one was sent three times, because these deployments sample their answers. The last two are a message with personal details, sent as written and after `redact()` removed the card number, the email, the phone and the account numbers.

Run the next cell.

In [ ]:
new = load_answers("new_messages")
seen = {}
for r in new:
    key = (r["model"], r["prompt"], r["split"], r["index"])
    seen.setdefault(key, {"message": r["message"], "answers": []})["answers"].append(f'{r["intent"]} ({r["confidence"]})')
for (model, prompt, kind, i), v in seen.items():
    if prompt == "v2":
        print(f"{model:11} {kind:17} {v['message'][:70]!r}\n            {' | '.join(v['answers'])}")

### Optional: a live call

The cell below sends one message to a live model through the OpenAI-compatible Chat Completions API: a free local model with [Ollama](https://ollama.com) (`http://localhost:11434/v1`, no key) or your own provider and key. It is **off** by default. To try it, set `LIVE = True` and the three settings. A hosted call costs money (about US$0.00023 per message on `chat-small` at the prices above); a local call is free but needs a computer that can run the model. Never type a key into a notebook that you share: read it from an environment variable. The course tested this route once with Gemma 3 4B on Ollama 0.40.1 on a Mac: "My card hasn't come yet. Where is it?" gave `card_arrival` with `high` confidence, in about 6 seconds including the model load.

In [ ]:
LIVE = False
BASE_URL = os.environ.get("ASSISTANT_BASE_URL", "http://localhost:11434/v1")
MODEL = os.environ.get("ASSISTANT_MODEL", "gemma3:4b")
API_KEY = os.environ.get("ASSISTANT_API_KEY", "not-needed")

if LIVE:
    %pip install -q "openai>=1"
    from openai import OpenAI
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY, max_retries=0, timeout=60)
    response = client.chat.completions.create(**build_request("My card hasn't come yet. Where is it?", MODEL, "v2"))
    print(parse(response.choices[0].message.content, response.choices[0].finish_reason))
else:
    print("Live call skipped. Set LIVE = True to try one.")

## Recap

- A prompt with the 77 intent names and a strict enum routes bank messages with **no training examples**: 87.5% on the test set for `chat-small`, against 89.0% for TF-IDF and 91.8% for fine-tuned DistilBERT, which both learned from 9,003 labelled messages.
- One line per intent (prompt v2) helped: compare prompts on the same fixed messages, message by message, with an interval.
- The errors sit on pairs of intents that the labels themselves blur.
- Cost and data: every message costs tokens and leaves the bank; the trained routers run on the bank's own server. The lesson page weighs when each is the better choice, and what may be sent to a hosted model.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Final assignment](https://learning.nextia-ai.com/courses/llm-apps/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/llm-apps/m07/classify-banking-requests/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). BANKING77 is CC BY 4.0 (PolyAI). The new messages are made up; the model answers are real recordings.